# 02 · No available budget - Table 9

Section 3.3: without money for pipe or pump substitutions ($B = 0$ in Eq. 46), the only
degrees of freedom are the valve head losses of the five hydraulic circuits, which redistribute
the cooling water towards the coolers with the highest penalties.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

from coolingnet import paper_case
from coolingnet.optimization import SolverSettings
from coolingnet.reporting import paper_reference as REF
from coolingnet.reporting import tables as T
from coolingnet.simulation import simulate_design
from coolingnet.workflow import PaperPipeline, PipelineConfig

RESULTS_DIR = Path("../results")
FORCE_RESOLVE = False

case = paper_case()
pipeline = PaperPipeline(case, PipelineConfig(results_dir=RESULTS_DIR, force=FORCE_RESOLVE, solver=SolverSettings(name="nlpbb", time_limit_s=600)))
sims = pipeline.run_simulations()
no_budget = pipeline.run_no_budget()
print(no_budget.report)

{'solver': 'nlpbb (ipopt nodes)', 'status': 'optimal', 'termination': 'optimal', 'objective': 88869.45931172569, 'bound': 88869.45931172569, 'gap': None, 'time_s': 0.20154079201165587, 'n_solutions': 1, 'message': 'multi-start NLP (5 solves); nodes=0 nlps=5; max constraint violation 7.53e-08'}


## Table 9 - optimization results with no available budget

In [2]:
table9 = T.table_no_budget(sims["current"], no_budget)
display(table9.round(2))
display(REF.compare(REF.TABLE9, table9.drop(index="Total")).round(2))

,Current cooling water flow rate (kg/s),Optimal cooling water flow rate (kg/s),Valve head loss (m),Penalty (kUSD/y),Penalty variation (%)
E-101,11.92,12.65,0.49,0.00,-100.00
E-102,28.85,23.04,3.70,27.29,37.67
E-103,35.57,33.69,1.90,24.86,9.52
E-104,15.53,17.21,0.00,16.25,-28.12
E-105,3.23,3.58,0.00,0.00,-100.00
Total,95.09,90.17,NaN,68.40,-7.21


,Current cooling water flow rate (kg/s) [paper],Current cooling water flow rate (kg/s) [this work],Optimal cooling water flow rate (kg/s) [paper],Optimal cooling water flow rate (kg/s) [this work],Valve head loss (m) [paper],Valve head loss (m) [this work],Penalty (kUSD/y) [paper],Penalty (kUSD/y) [this work],Penalty variation (%) [paper],Penalty variation (%) [this work]
Heat exchanger,,,,,,,,,,
E-101,11.92,11.92,12.65,12.65,0.49,0.49,0.00,0.00,-100.00,-100.00
E-102,28.85,28.85,23.04,23.04,3.70,3.70,27.29,27.29,37.69,37.67
E-103,35.57,35.57,33.69,33.69,1.90,1.90,24.86,24.86,9.52,9.52
E-104,15.53,15.53,17.21,17.21,0.00,0.00,16.25,16.25,-28.13,-28.12
E-105,3.23,3.23,3.58,3.58,0.00,0.00,0.00,0.00,-100.00,-100.00


In [3]:
cur, opt = sims["current"].costs, no_budget.costs
summary = pd.DataFrame(
    {
        "current condition": [cur["cpen"] / 1e3, cur["cope"] / 1e3, (cur["cpen"] + cur["cope"]) / 1e3],
        "valve manipulation": [opt["cpen"] / 1e3, opt["cope"] / 1e3, (opt["cpen"] + opt["cope"]) / 1e3],
        "paper": [REF.NO_BUDGET_TEXT["penalty_before"], REF.NO_BUDGET_TEXT["operating_before"], None],
        "paper (after)": [REF.NO_BUDGET_TEXT["penalty_after"], REF.NO_BUDGET_TEXT["operating_after"], None],
    },
    index=["penalty (kUSD/y)", "operating cost (kUSD/y)", "total (kUSD/y)"],
)
display(summary.round(2))
reduction = (1 - (opt["cpen"] + opt["cope"]) / (cur["cpen"] + cur["cope"])) * 100
print(f"total cost reduction: {reduction:.2f} %  (paper: {REF.NO_BUDGET_TEXT['total_reduction_pct']} %)")

,current condition,valve manipulation,paper,paper (after)
penalty (kUSD/y),73.71,68.40,73.71,68.40
operating cost (kUSD/y),20.16,20.47,20.16,20.47
total (kUSD/y),93.87,88.87,NaN,NaN


total cost reduction: 5.33 %  (paper: 5.33 %)


## Independent cross-check with the NumPy simulator

With the design fixed, the problem has only five continuous unknowns (the valve head losses).
Minimizing operating + penalty cost over them with SciPy (SLSQP) on top of the NumPy simulator
must give the same optimum as the Pyomo/IPOPT solution.

In [4]:
pen = case.heat_exchanger_defaults.penalty_factor_USD_per_C_y
factor = case.scenarios.hot_flow_increase_factor


def total_cost(valves: np.ndarray) -> float:
    r = simulate_design(case, valve_head_losses_m={i + 1: max(v, 0.0) for i, v in enumerate(valves)}, hot_flow_factor=factor)
    return r.total_cost_USD_y


x0 = np.zeros(5)
res = minimize(total_cost, x0, method="SLSQP", bounds=[(0, 10)] * 5, options={"ftol": 1e-10, "maxiter": 300})
print("SLSQP valve head losses (m):", np.round(res.x, 2), " total cost:", round(res.fun / 1e3, 2), "kUSD/y")
print("Pyomo/IPOPT valve head losses (m):", np.round(no_budget.circuits_df()["valve_head_loss_m"].values, 2), " total cost:", round(opt["total_cost"] / 1e3, 2), "kUSD/y")

SLSQP valve head losses (m): [0.49 3.69 1.91 0.   0.  ]  total cost: 88.87 kUSD/y
Pyomo/IPOPT valve head losses (m): [0.49 3.7  1.9  0.   0.  ]  total cost: 88.87 kUSD/y
